In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
len(documents)

1368

In [4]:
documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [5]:
documents = documents_llm

In [6]:
# sample doc
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
# With structured output, we ask the LLM to return data in a specific format instead of free-form text. 
# For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
# responses.parse() is used to enforce a structure
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-kDWwal0kvt52bY6t38qZvc0w on requests per day (RPD): Limit 50, Used 50, Requested 1. Please try again in 28m48s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
result = response.output_parsed
print(result)

In [ ]:
print(result.questions)

In [ ]:
from evaluation_utils import llm_structured, calc_price

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
usage.input_tokens, usage.output_tokens

In [ ]:
cost = calc_price(usage)
cost

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

In [ ]:
import pandas as pd
pd.DataFrame(records)

In [ ]:
# add records for all the documents

from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

from tqdm.auto import tqdm

ground_truth = []
usages = []

# try it for the first 5 documents
for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

In [ ]:
pd.DataFrame(ground_truth)

In [ ]:
len(ground_truth)

In [ ]:
# Running the calls one after another wastes most of the time waiting on the network. 
# Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. 
# We process the documents in parallel and track progress while the requests run.

# One caution: don't open too many connections at once, or you'll hit the provider's rate limits. 
# Five or six workers is a safe default here.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth), len(documents)*5

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)